In [2]:
import sys
sys.path.append('..')  # Sube un nivel
from utils import *

In [3]:
df_imputed = pd.read_csv(CFG.path_df_imputed_corrected)
print(f"Dataset shape: {df_imputed.shape}")
df_imputed.columns = clean_feature_names(df_imputed.columns)
print(f"\nColumns: {list(df_imputed.columns)}")
df_imputed.tail()

Dataset shape: (2292, 33)

Columns: ['Plant_Height (cm)', 'Chlorophyll (SPAD)', 'Number of Flowers', 'Number of Harvested Fruits', 'Weight of Harvested Fruits (Kg)', 'Fruit Height (mm)', 'Sap pH', 'Sap K (ppm)', 'Sap Ca (ppm)', 'Sap Na (ppm)', 'Sap NO3 (ppm)', 'Sap Conductivity (mS/cm)', 'Soil pH Horiba', 'Soil K Horiba (ppm)', 'Soil Ca Horiba (ppm)', 'Soil Na Horiba (ppm)', 'Soil NO3 Horiba (ppm)', 'Soil Conductivity Horiba (mS/cm)', 'Fruit Diameter (mm)', '7in1_Ph(pH)', '7in1_Moisture(%RH)', '7in1_S_Temperature(C)', 'Air_sensor_Temperature(C)', 'Air_sensor_Humidity(%RH)', '7in1_Conductivity(mS/cm)', 'Pynamometer_Radiation(W/m2)', 'Nitrogen', 'Phosphorus', 'Potassium', 'Year', 'Month', 'Day', 'Treatment_Num']


,Plant_Height (cm),Chlorophyll (SPAD),Number of Flowers,Number of Harvested Fruits,Weight of Harvested Fruits (Kg),Fruit Height (mm),Sap pH,Sap K (ppm),Sap Ca (ppm),Sap Na (ppm),...,Air_sensor_Humidity(%RH),7in1_Conductivity(mS/cm),Pynamometer_Radiation(W/m2),Nitrogen,Phosphorus,Potassium,Year,Month,Day,Treatment_Num
2287,171.0,57.33,23.6,1.0,0.180,70.3,5.438,3580.0,385.4,49.60,...,79.64,0.1307,271.1,2.0,2.0,2.0,2024.0,12.0,4.0,27.0
2288,167.2,57.19,23.4,7.0,0.706,60.7,5.438,3230.0,263.2,49.70,...,68.08,0.1242,262.7,2.0,2.0,2.0,2024.0,12.0,4.0,27.0
2289,136.6,60.03,14.1,6.0,0.829,70.2,5.506,3490.0,226.0,57.10,...,63.30,0.0280,269.0,0.0,0.0,1.0,2024.0,12.0,4.0,4.0
2290,64.6,50.57,7.7,2.0,0.174,50.7,5.667,3330.0,223.6,50.35,...,53.07,0.2098,347.2,0.0,0.0,1.0,2024.0,12.0,4.0,4.0
2291,180.5,59.77,19.7,1.0,0.150,70.1,5.594,3480.0,181.2,57.40,...,64.70,0.0144,263.5,0.0,0.0,1.0,2024.0,12.0,4.0,4.0


In [4]:
with open(CFG.treat_quantiles_path, 'r') as f:
    treatments_quantile_unified = json.load(f)

In [5]:
df_imputed, df_imputed['target'] = codificar_clase_cuartiles(df_imputed, filter_data=True)

In [6]:
json_best_variables = Rf"{CFG.Root}\Resultados\classification_exclude_prod\most_frequent_variables_80.json"
# TODO: Checkear que la ruta existe. Se debió ejecutar el modelo de train anterior.
list_best_vars = read_best_variables(json_best_variables)

In [9]:
df_imputed['Fecha'] = pd.to_datetime(df_imputed[['Year', 'Month', 'Day']])
fecha_cosecha = pd.to_datetime('2024-10-02')

In [10]:
# iterar por cada variable
# Crear lista para almacenar resultados
results = []

# iterar por cada variable
for var in list_best_vars:
    # comparar la media y revisar si tiene diferentes std antes y despues de fecha_cosecha de la variable
    # comaprar entre grupos de target 0 y 1
    before_cosecha_group1 = df_imputed[(df_imputed['target'] == 0) & (df_imputed['Fecha'] < fecha_cosecha)][var]
    after_cosecha_group1 = df_imputed[(df_imputed['target'] == 0) & (df_imputed['Fecha'] >= fecha_cosecha)][var]
    before_cosecha_group2 = df_imputed[(df_imputed['target'] == 1) & (df_imputed['Fecha'] < fecha_cosecha)][var]
    after_cosecha_group2 = df_imputed[(df_imputed['target'] == 1) & (df_imputed['Fecha'] >= fecha_cosecha)][var]

    print(f"\nVariable: {var}")
    print(f" Target 0 - Before harvest: mean={before_cosecha_group1.mean():.4f}, std={before_cosecha_group1.std():.4f}, n={len(before_cosecha_group1)}")
    print(f" Target 0 - After harvest: mean={after_cosecha_group1.mean():.4f}, std={after_cosecha_group1.std():.4f}, n={len(after_cosecha_group1)}")
    print(f" Target 1 - Before harvest: mean={before_cosecha_group2.mean():.4f}, std={before_cosecha_group2.std():.4f}, n={len(before_cosecha_group2)}")
    print(f" Target 1 - After harvest: mean={after_cosecha_group2.mean():.4f}, std={after_cosecha_group2.std():.4f}, n={len(after_cosecha_group2)}")

    results.append({
        'Variable': var,
        'Target0_Before_mean': before_cosecha_group1.mean(),
        'Target0_Before_std': before_cosecha_group1.std(),
        'Target0_Before_n': len(before_cosecha_group1),
        'Target0_After_mean': after_cosecha_group1.mean(),
        'Target0_After_std': after_cosecha_group1.std(),
        'Target0_After_n': len(after_cosecha_group1),
        'Target1_Before_mean': before_cosecha_group2.mean(),
        'Target1_Before_std': before_cosecha_group2.std(),
        'Target1_Before_n': len(before_cosecha_group2),
        'Target1_After_mean': after_cosecha_group2.mean(),
        'Target1_After_std': after_cosecha_group2.std(),
        'Target1_After_n': len(after_cosecha_group2)
    })

df_results = pd.DataFrame(results)
df_results.to_csv('productivity_comparison_mean.csv', index=False)
print(f"\n\nResultados guardados en productivity_comparison_mean.csv")
df_results


Variable: Sap Na (ppm)
 Target 0 - Before harvest: mean=48.1564, std=9.6459, n=133
 Target 0 - After harvest: mean=50.7790, std=8.2974, n=480
 Target 1 - Before harvest: mean=44.1541, std=7.5880, n=133
 Target 1 - After harvest: mean=49.1114, std=7.0556, n=479

Variable: Soil NO3 Horiba (ppm)
 Target 0 - Before harvest: mean=115.2912, std=109.1881, n=133
 Target 0 - After harvest: mean=46.0291, std=40.7801, n=480
 Target 1 - Before harvest: mean=150.1843, std=146.4520, n=133
 Target 1 - After harvest: mean=46.6363, std=64.2698, n=479

Variable: Sap Conductivity (mS/cm)
 Target 0 - Before harvest: mean=11.1861, std=1.0686, n=133
 Target 0 - After harvest: mean=10.8238, std=1.1447, n=480
 Target 1 - Before harvest: mean=11.1811, std=1.1588, n=133
 Target 1 - After harvest: mean=10.6094, std=1.1070, n=479

Variable: Soil pH Horiba
 Target 0 - Before harvest: mean=6.6364, std=0.4230, n=133
 Target 0 - After harvest: mean=6.5982, std=0.3157, n=480
 Target 1 - Before harvest: mean=6.6305, s

,Variable,Target0_Before_mean,Target0_Before_std,Target0_Before_n,Target0_After_mean,Target0_After_std,Target0_After_n,Target1_Before_mean,Target1_Before_std,Target1_Before_n,Target1_After_mean,Target1_After_std,Target1_After_n
0,Sap Na (ppm),48.156391,9.645914,133,50.778958,8.297433,480,44.154135,7.587994,133,49.111378,7.055563,479
1,Soil NO3 Horiba (ppm),115.291203,109.188134,133,46.029096,40.780082,480,150.184331,146.451973,133,46.636332,64.269795,479
2,Sap Conductivity (mS/cm),11.186098,1.068604,133,10.823780,1.144696,480,11.181128,1.158804,133,10.609390,1.107024,479
3,Soil pH Horiba,6.636383,0.423006,133,6.598204,0.315684,480,6.630496,0.391814,133,6.759643,0.269184,479
4,Sap Ca (ppm),244.824436,117.078053,133,309.767083,150.604919,480,268.264662,144.502294,133,297.738413,159.334172,479
5,Chlorophyll (SPAD),50.359023,6.359124,133,54.695229,4.300790,480,50.929699,6.103173,133,56.592380,3.357572,479
6,Sap NO3 (ppm),1571.245113,628.497646,133,733.085625,445.712678,480,1884.899248,562.634011,133,884.486013,682.440485,479
7,7in1_S_Temperature(C),27.609549,2.174025,133,25.816312,1.764759,480,27.287368,2.258539,133,24.869812,1.548263,479
8,Soil Ca Horiba (ppm),81.744486,50.759346,133,47.621424,39.060266,480,80.453008,57.477860,133,37.676409,44.710646,479
9,Air_sensor_Humidity(%RH),59.048195,7.247786,133,66.290604,8.093058,480,60.009323,8.616562,133,69.111253,6.799270,479
